In [1]:
### misc
import pandas as pd
import numpy as np
import os
from pathlib import Path
import pickle
import time
from itertools import product

#### graphical
import matplotlib.pyplot as plt
import corner

#### ML
import sklearn
from sklearn.decomposition import PCA
import tensorflow as tf
import keras
from keras import layers

from WMSE import WMSE, WMSE_metric

##### poke gpu
os.environ["CUDA_VISIBLE_DEVICES"]="1"

physical_devices = tf.config.list_physical_devices("GPU") 

tf.config.experimental.set_memory_growth(physical_devices[0], True)

gpu0usage = tf.config.experimental.get_memory_info("GPU:0")["current"]

print("Current GPU usage:\n"
     + " - GPU0: " + str(gpu0usage) + "B\n")

modelpath = '/home/hatte/M4/models'

2025-04-08 14:54:49.753921: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744120489.765776 1711050 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744120489.769285 1711050 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-08 14:54:49.782724: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Current GPU usage:
 - GPU0: 0B



I0000 00:00:1744120491.141289 1711050 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 12696 MB memory:  -> device: 0, name: NVIDIA RTX A4500, pci bus id: 0000:61:00.0, compute capability: 8.6


In [2]:
def scheduler(epoch, lr,):
    ## Learning rate scheduler
    # Decreases learning rate in-training for stability
    if lr < 1e-5:
        return float(lr)
    else:
        return float(lr * tf.math.exp(-6e-6))

In [3]:
df_full = pd.read_hdf('../grids/Chiara.hdf5', key='df') ## edit for your grid!!

df_full['logLPhot'] = np.log10(df_full['LPhot'])

df_full['lognumax'] = np.log10(df_full['numax']*3090)

df_full['logdnuSer'] = np.log10(df_full['dnuSer']*135)

df_full['logAge'] = np.log10(df_full['age'])

#### define inputs
inputs = ['massini', 'zini', 'yini', 'alphaMLT', 'logAge', 'eta', 'alphaFe']

#### define outputs
classical_outputs = ['FeH', 'logLPhot', 'Teff']
astero_outputs = ['lognumax', 'logdnuSer'] 

outputs = classical_outputs+astero_outputs

df = df_full[inputs+outputs]

df_norm = (df - df.min())/(df.max() - df.min())

## check df_norm.describe looks reasonable (min=0, max=1):
df_norm.describe()

#### train/test split with seed 
seed = 42

df_train = df_norm.sample(frac=0.95, random_state=seed)
df_test = df_norm.drop(df_train.index)

df_train_inputs, df_val_inputs, df_train_outputs, df_val_outputs = sklearn.model_selection.train_test_split(df_train[inputs],df_train[outputs], test_size = 0.05, random_state=seed)

print("Training set: ", len(df_train_inputs))
print("Validation set: ", len(df_val_inputs))
print("Test set: ", len(df_test))

Training set:  6754081
Validation set:  355478
Test set:  374187


In [4]:
#unnormed_weights_dict = {'FeH':0.01, 'logLPhot':0.001, 'Teff':1, 'numax':0.001/3090, 'dnuSer':0.0001/135}

unnormed_weights_dict = {'FeH':0.01, 'logLPhot':0.001, 'Teff':1, 'lognumax':0.0001, 'logdnuSer':0.0001}

unnormed_weights = list(unnormed_weights_dict.values())

weights = [2*unnormed_weights_dict[i]/(df[i].max() - df[i].min()) for i in outputs]

In [5]:
n_dense_layers = 6

dense_layer_units = 128

Nepochs = 5000

learning_rate = 0.0001

model_name = 'logAge-logLPhot-lognumax-logdnuSer-exponent-6e-6-Adam'

loss_func = 'WMSE'

df_train_inputs.join(df_train_outputs).to_hdf(f'{modelpath}/long-runs/training-data/training-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.hdf5', key = 'df')

In [6]:
checkpoint_dir = f'{modelpath}/long-runs/checkpoint/chk-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.model.keras'

full_model_dir = f'{modelpath}/long-runs/full-model/mod-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}/'

if not os.path.exists(full_model_dir):
    os.makedirs(full_model_dir)

historyfile = f'{modelpath}/long-runs/history/hist-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.json'
    
cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath = checkpoint_dir, verbose = 1, save_best_only = True, save_freq = 'epoch')

lr_callback = tf.keras.callbacks.LearningRateScheduler(scheduler, )

In [ ]:
######## map out model architecture
#### input layer
nn_input = keras.Input(shape=(len(inputs),))

#### dense layer(s)
for n_dense_layer in range(n_dense_layers):
    if n_dense_layer == 0:
        dense_layer = layers.Dense(dense_layer_units, activation='elu')(nn_input)
    else:
        dense_layer = layers.Dense(dense_layer_units, activation='elu')(dense_layer)

#### output layer
nn_output =  layers.Dense(len(outputs), activation='linear')(dense_layer)

######## store architecture as keras model
model = keras.Model(inputs=nn_input, outputs=nn_output, name=model_name)

tb_callback = tf.keras.callbacks.TensorBoard(log_dir = f'{modelpath}/logs/long-runs/log-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}')

model.compile(loss=WMSE(weights), optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate))

history = model.fit(df_train_inputs,
          df_train_outputs,
          validation_data=(df_val_inputs,df_val_outputs),
          batch_size=2**14, #change higher
          verbose=1,
          epochs=Nepochs,
          shuffle=True, callbacks = [tb_callback, cp_callback, lr_callback]) 

tf.saved_model.save(model, full_model_dir)
hist_df = pd.DataFrame(history.history)
    
with open(os.path.join(modelpath, historyfile), mode="w") as f:
    hist_df.to_json(f)

Epoch 1/5000


I0000 00:00:1744120502.259852 1711934 service.cc:148] XLA service 0x7277b800f540 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1744120502.259883 1711934 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A4500, Compute Capability 8.6
2025-04-08 14:55:02.295989: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1744120502.397675 1711934 cuda_dnn.cc:529] Loaded cuDNN version 90300
2025-04-08 14:55:02.455415: W external/local_xla/xla/service/gpu/nvptx_compiler.cc:930] The NVIDIA driver's CUDA version is 12.2 which is older than the PTX compiler version 12.5.82. Because the driver is older than the PTX compiler version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.
2025-04-08 14:55:02.92662

 54/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 1721661.8750

I0000 00:00:1744120503.535962 1711934 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


395/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1441554.1250

2025-04-08 14:55:05.421074: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_490', 336 bytes spill stores, 480 bytes spill loads

2025-04-08 14:55:05.557620: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_490', 452 bytes spill stores, 452 bytes spill loads

2025-04-08 14:55:05.672592: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_490', 4244 bytes spill stores, 4228 bytes spill loads



413/413 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1438315.6250
Epoch 1: val_loss improved from inf to 1340456.37500, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-logAge-logLPhot-lognumax-logdnuSer-exponent-6e-6-Adam-nlayers-6-nunits-128-epochs-5000-lrate-0.0001-lossfunc-WMSE.model.keras
413/413 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 1438142.6250 - val_loss: 1340456.3750 - learning_rate: 9.9999e-05
Epoch 2/5000
408/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1341228.0000
Epoch 2: val_loss improved from 1340456.37500 to 1332746.37500, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-logAge-logLPhot-lognumax-logdnuSer-exponent-6e-6-Adam-nlayers-6-nunits-128-epochs-5000-lrate-0.0001-lossfunc-WMSE.model.keras
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 1341188.5000 - val_loss: 1332746.3750 - learning_rate: 9.9999e-05
Epoch 3/5000
410/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1330947.2500
Epoch 3: val_loss improved from 1332746.37500 to 1312999.37500

In [10]:
unnormed_weights_dict = {'FeH':0.01, 'logLPhot':0.001, 'Teff':1, 'lognumax':0.0001, 'logdnuSer':0.0001}

unnormed_weights = list(unnormed_weights_dict.values())

weights = [2*unnormed_weights_dict[i]/(df[i].max() - df[i].min()) for i in outputs]

In [8]:
custom_objects =  {'WMSE':WMSE_metric}

model= tf.keras.models.load_model(checkpoint_dir, custom_objects = custom_objects)


In [13]:
n_learning_rate = model.optimizer.get_config()['learning_rate']

nnepochs = nepochs + 90000

tb_callback = tf.keras.callbacks.TensorBoard(log_dir = f'{modelpath}/logs/long-runs/log-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}')

model.compile(loss=WMSE(weights), optimizer=tf.keras.optimizers.Adam(learning_rate=n_learning_rate))

In [ ]:
history = model.fit(df_train_inputs,
          df_train_outputs,
          validation_data=(df_val_inputs,df_val_outputs),
          batch_size=2**14, #change higher
          verbose=1,
          epochs=nnepochs,
          shuffle=True, callbacks = [tb_callback, cp_callback, lr_callback],
          initial_epoch=nepochs)


Epoch 10001/100000
413/413 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1133124.5000
Epoch 10001: val_loss did not improve from 5473.95312
413/413 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 1131305.1250 - val_loss: 19908.8555 - learning_rate: 9.4159e-05
Epoch 10002/100000
407/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 16662.3145
Epoch 10002: val_loss did not improve from 5473.95312
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 16631.5918 - val_loss: 12541.6953 - learning_rate: 9.4158e-05
Epoch 10003/100000
400/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 11958.8096
Epoch 10003: val_loss did not improve from 5473.95312
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 11944.0664 - val_loss: 10892.0996 - learning_rate: 9.4158e-05
Epoch 10004/100000
399/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 10589.1953
Epoch 10004: val_loss did not improve from 5473.95312
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 10580.6748 - val_loss: 10120.1777 - learning_rate: 9.4157e-05
Epoch 10005/100000
4

In [ ]:
n_learning_rate = model.optimizer.get_config()['learning_rate']

nepochs = 5000 +Nepochs

tb_callback = tf.keras.callbacks.TensorBoard(log_dir = f'{modelpath}/logs/long-runs/log-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}')

model.compile(loss=WMSE(weights), optimizer=tf.keras.optimizers.Adam(learning_rate=n_learning_rate))

In [ ]:
history = model.fit(df_train_inputs,
          df_train_outputs,
          validation_data=(df_val_inputs,df_val_outputs),
          batch_size=2**14, #change higher
          verbose=1,
          epochs=nepochs,
          shuffle=True, callbacks = [tb_callback, cp_callback, lr_callback],
          initial_epoch=Nepochs)